# LeetCode #207 — Course Schedule: phát hiện chu trình trong đồ thị có hướng (sắp xếp tô-pô)

**Mức độ:** Medium · **Chủ đề:** Depth-First Search, Breadth-First Search, Graph, Topological Sort

Skeleton trên LeetCode:

```python
class Solution:
    def canFinish(self, numCourses: int, prerequisites: List[List[int]]) -> bool:
```

Mục tiêu của notebook:

1. Mô hình hóa: môn học là **đỉnh**, quan hệ "phải học `b` trước `a`" là **cạnh có hướng** `b → a`.
2. Học xong mọi môn ⟺ đồ thị **không có chu trình** (là một DAG).
3. Cách 1 — **Kahn (BFS)**: liên tục lấy các môn **không còn môn tiên quyết nào** (bậc vào = 0).
4. Cách 2 — **DFS ba màu**: gặp lại một đỉnh **đang trên đường đi hiện tại** ⟹ có chu trình.

# 1. Bài toán là gì?

Có `numCourses` môn học đánh số `0 .. numCourses − 1`. Mảng `prerequisites` gồm các cặp `[a, b]` nghĩa là **phải học `b` trước khi học `a`**.

Trả về `True` nếu có thể học hết mọi môn, ngược lại `False`.

**Constraints (theo đề):**

- `1 <= numCourses <= 2000`
- `0 <= prerequisites.length <= 5000`
- `prerequisites[i].length == 2`, `0 <= a, b < numCourses`
- Các cặp `prerequisites[i]` **khác nhau**.

---

## Ví dụ

```text
numCourses = 2, prerequisites = [[1, 0]]          ->  True    (học 0 rồi 1)
numCourses = 2, prerequisites = [[1, 0], [0, 1]]  ->  False   (0 cần 1, 1 cần 0: vòng lặp)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Đồ thị (graph) và cách biểu diễn
2. Duyệt theo chiều rộng (BFS — Breadth-First Search)
3. Duyệt theo chiều sâu (DFS — Depth-First Search)
4. Sắp xếp tô-pô (topological sort) và phát hiện chu trình

Cuối mục có phần **2.5 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Đồ thị (graph) và cách biểu diễn

### Đồ thị là gì?

Đồ thị gồm các **đỉnh** (vertex/node) và các **cạnh** (edge) nối cặp đỉnh. Cạnh có thể **vô hướng** (đi được hai chiều, như đường hai chiều) hoặc **có hướng**. Cây là trường hợp riêng: đồ thị liên thông, không chu trình.

```text
1 ── 2
│    │       đỉnh: 1, 2, 3, 4
4 ── 3       cạnh: (1,2) (2,3) (3,4) (4,1)  → có chu trình 1-2-3-4-1
```

### Ba cách biểu diễn

| Cách | Mô tả | Hợp khi |
|---|---|---|
| Danh sách cạnh | `[(1,2), (2,3), …]` | đầu vào thô |
| **Danh sách kề** | `{1: [2, 4], 2: [1, 3], …}` | hầu hết bài duyệt (BFS/DFS) |
| Ma trận kề | `M[u][v] = 1` nếu có cạnh | đồ thị dày, cần hỏi "có cạnh u-v?" nhanh |
| Đối tượng nút | `node.neighbors = [...]` | LeetCode kiểu "Clone Graph" |

In [1]:
from collections import defaultdict, deque

canh = [(1, 2), (2, 3), (3, 4), (4, 1)]
ke = defaultdict(list)
for u, v in canh:                # vô hướng: thêm cả hai chiều
    ke[u].append(v)
    ke[v].append(u)
print("danh sách kề:", dict(ke))

n = 4
M = [[0] * (n + 1) for _ in range(n + 1)]
for u, v in canh:
    M[u][v] = M[v][u] = 1
print("ma trận kề (bỏ hàng/cột 0):")
for row in M[1:]:
    print("  ", row[1:])

danh sách kề: {1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [3, 1]}
ma trận kề (bỏ hàng/cột 0):
   [0, 1, 0, 1]
   [1, 0, 1, 0]
   [0, 1, 0, 1]
   [1, 0, 1, 0]


### Đồ thị dạng đối tượng nút

Mỗi nút giữ danh sách **tham chiếu** tới các nút hàng xóm. Vì có chu trình, khi duyệt **bắt buộc** phải nhớ nút đã thăm — nếu không sẽ đi vòng mãi.

In [2]:
class GNode:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []

nodes = {i: GNode(i) for i in range(1, 5)}
for u, v in canh:
    nodes[u].neighbors.append(nodes[v])
    nodes[v].neighbors.append(nodes[u])

def duyet_nut(start):
    seen, q, out = {start}, deque([start]), []
    while q:
        x = q.popleft()
        out.append(x.val)
        for y in x.neighbors:
            if y not in seen:
                seen.add(y); q.append(y)
    return out

print("BFS từ nút 1:", duyet_nut(nodes[1]))
print("hàng xóm của 3:", [y.val for y in nodes[3].neighbors])

BFS từ nút 1: [1, 2, 4, 3]
hàng xóm của 3: [2, 4]


### Sao chép **nông** và sao chép **sâu**

- Sao chép nông: tạo đối tượng mới nhưng vẫn **trỏ tới** các nút cũ.
- Sao chép sâu: tạo **mọi** nút mới, và cạnh mới nối các nút **mới** với nhau; không chia sẻ gì với bản gốc.

Với đồ thị có chu trình, sao chép sâu cần một **bảng `cũ → mới`** để mỗi nút cũ chỉ được tạo bản sao **một lần** và các cạnh trỏ đúng bản sao.

In [3]:
import copy
a_list = [[1, 2], [3]]
nong = list(a_list)
sau = copy.deepcopy(a_list)
a_list[0].append(99)
print("gốc:", a_list, "| nông (bị ảnh hưởng):", nong, "| sâu:", sau)

gốc: [[1, 2, 99], [3]] | nông (bị ảnh hưởng): [[1, 2, 99], [3]] | sâu: [[1, 2], [3]]


## 2.2 Duyệt theo chiều rộng (BFS — Breadth-First Search)

### BFS là gì?

BFS duyệt đồ thị/cây **theo từng lớp**: thăm điểm xuất phát, rồi **mọi** điểm cách 1 bước, rồi mọi điểm cách 2 bước, … giống vết sóng lan ra khi ném hòn đá xuống nước.

Công cụ: **hàng đợi** (FIFO). Điểm phát hiện trước được xử lý trước → các điểm gần luôn được xử lý trước các điểm xa.

```text
        A            lớp 0: A
       / \           lớp 1: B, C
      B   C          lớp 2: D, E, F
     / \   \
    D   E   F        thứ tự BFS: A B C D E F
```

### Khung BFS trên đồ thị

```text
q = deque([start]); visited = {start}
while q:
    u = q.popleft()
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            visited.add(v)        # đánh dấu NGAY khi đưa vào hàng đợi
            q.append(v)
```

In [4]:
from collections import deque

do_thi = {                       # danh sách kề
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def bfs(graph, start):
    q, visited, thu_tu = deque([start]), {start}, []
    while q:
        u = q.popleft()
        thu_tu.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
        print(f"lấy {u} | hàng đợi: {list(q)}")
    return thu_tu

print("Thứ tự BFS:", bfs(do_thi, "A"))

lấy A | hàng đợi: ['B', 'C']
lấy B | hàng đợi: ['C', 'D', 'E']
lấy C | hàng đợi: ['D', 'E', 'F']
lấy D | hàng đợi: ['E', 'F']
lấy E | hàng đợi: ['F']
lấy F | hàng đợi: []
Thứ tự BFS: ['A', 'B', 'C', 'D', 'E', 'F']


### BFS theo tầng (level-order) trên cây nhị phân

Muốn biết nút nào thuộc tầng nào: ở đầu mỗi vòng, hàng đợi chứa **đúng** các nút của một tầng. Ghi lại `len(q)`, lấy ra đúng chừng ấy nút (và đẩy con của chúng vào) — đó là một tầng.

In [5]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_bfs = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))

def theo_tang(root):
    if not root:
        return []
    q, tang = deque([root]), []
    while q:
        cur = []
        for _ in range(len(q)):          # đúng số nút của tầng hiện tại
            node = q.popleft()
            cur.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        tang.append(cur)
    return tang

print("các tầng:", theo_tang(cay_bfs))

các tầng: [[3], [9, 20], [15, 7]]


### BFS tìm đường ngắn nhất trên lưới

Trên đồ thị **không trọng số** (mỗi bước dài như nhau), lần đầu BFS chạm tới một ô chính là đi bằng **số bước ít nhất**. Lưu `dist` khi đưa ô vào hàng đợi.

In [6]:
luoi = ["S.#.",
        "..#.",
        "...E"]

def ngan_nhat(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1

print("số bước ít nhất từ S tới E:", ngan_nhat(luoi))

số bước ít nhất từ S tới E: 5


### Độ phức tạp và lỗi hay gặp

`O(V + E)` thời gian (mỗi đỉnh vào hàng đợi một lần, mỗi cạnh xét một/hai lần), `O(V)` bộ nhớ. Trên cây: `O(n)`; trên lưới `R×C`: `O(R·C)`.

- Đánh dấu `visited` khi **lấy ra** thay vì khi **đưa vào** → một ô có thể vào hàng đợi nhiều lần (chậm, đếm sai).
- Dùng `list.pop(0)` → mỗi lần `O(n)`.
- Dùng `pop()` (cuối) → thành DFS, mất tính "gần trước".
- Quên kiểm tra biên lưới.

## 2.3 Duyệt theo chiều sâu (DFS — Depth-First Search)

### DFS là gì?

DFS đi **sâu nhất có thể** theo một nhánh, tới khi hết đường thì **quay lui** về ngã rẽ gần nhất và thử nhánh khác — giống đi mê cung bằng cách luôn bám một bức tường.

```text
        A
       / \          DFS từ A (thử hàng xóm theo thứ tự):
      B   C         A → B → D (cụt, quay lui) → E → F → C
     / \   \
    D   E - F
```

Công cụ: **đệ quy** (dùng call stack) hoặc **ngăn xếp** tự quản lý.

### Khung DFS

```text
def dfs(u):
    visited.add(u)
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            dfs(v)
```

In [7]:
do_thi_dfs = {
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def dfs_de_quy(graph, u, visited, thu_tu, sau=0):
    visited.add(u)
    thu_tu.append(u)
    print("    " * sau + f"vào {u}")
    for v in graph[u]:
        if v not in visited:
            dfs_de_quy(graph, v, visited, thu_tu, sau + 1)
    print("    " * sau + f"rời {u}")

tt = []
dfs_de_quy(do_thi_dfs, "A", set(), tt)
print("Thứ tự DFS:", tt)

vào A
    vào B
        vào D
        rời D
        vào E
            vào F
                vào C
                rời C
            rời F
        rời E
    rời B
rời A
Thứ tự DFS: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS bằng ngăn xếp (không đệ quy)

Thay hàng đợi của BFS bằng **ngăn xếp**. Tránh được giới hạn độ sâu đệ quy khi dữ liệu lớn.

In [8]:
def dfs_ngan_xep(graph, start):
    st, visited, thu_tu = [start], set(), []
    while st:
        u = st.pop()
        if u in visited:
            continue
        visited.add(u)
        thu_tu.append(u)
        for v in reversed(graph[u]):        # đảo để hàng xóm đầu tiên được lấy ra trước
            if v not in visited:
                st.append(v)
    return thu_tu

print("DFS bằng ngăn xếp:", dfs_ngan_xep(do_thi_dfs, "A"))

DFS bằng ngăn xếp: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS trên lưới: tô loang (flood fill) và đếm vùng

Lưới là đồ thị ẩn: mỗi ô nối với 4 ô kề. DFS từ một ô sẽ "loang" khắp vùng liên thông chứa nó. Đếm số lần phải bắt đầu DFS mới = số vùng.

In [9]:
def dem_dao(grid):
    g = [list(row) for row in grid]
    R, C = len(g), len(g[0])
    def loang(r, c):
        if r < 0 or r >= R or c < 0 or c >= C or g[r][c] != "1":
            return                              # ra ngoài / nước / đã thăm
        g[r][c] = "#"                           # đánh dấu đã thăm
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            loang(r + dr, c + dc)
    dem = 0
    for r in range(R):
        for c in range(C):
            if g[r][c] == "1":
                dem += 1
                loang(r, c)
    return dem, ["".join(row) for row in g]

print(dem_dao(["11000",
               "11010",
               "00011"]))

(2, ['##000', '##0#0', '000##'])


### DFS trên cây: mang thông tin dọc đường đi

Trên cây không cần `visited` (không có chu trình). DFS hay **truyền xuống** thông tin của đường đi từ gốc (tổng, đường đi hiện tại) và **trả lên** kết quả của cây con.

In [10]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_dfs = TreeNode(5, TreeNode(4, TreeNode(11, TreeNode(7), TreeNode(2))), TreeNode(8, TreeNode(13), TreeNode(4)))

def moi_duong_goc_la(node, duong, out):
    if node is None:
        return
    duong.append(node.val)                     # đi xuống: thêm nút
    if node.left is None and node.right is None:
        out.append((list(duong), sum(duong)))  # tới lá: ghi lại một bản sao
    moi_duong_goc_la(node.left, duong, out)
    moi_duong_goc_la(node.right, duong, out)
    duong.pop()                                # quay lên: bỏ nút

ket_qua = []
moi_duong_goc_la(cay_dfs, [], ket_qua)
for d, s in ket_qua:
    print(d, "tổng =", s)

[5, 4, 11, 7] tổng = 27
[5, 4, 11, 2] tổng = 22
[5, 8, 13] tổng = 26
[5, 8, 4] tổng = 17


### So sánh BFS và DFS

| | BFS | DFS |
|---|---|---|
| Cấu trúc | hàng đợi | ngăn xếp / đệ quy |
| Thứ tự | theo lớp (gần → xa) | theo nhánh (sâu trước) |
| Đường ngắn nhất (không trọng số) | **có** | không đảm bảo |
| Bộ nhớ | độ rộng lớn nhất | độ sâu lớn nhất |
| Hợp với | khoảng cách, theo tầng | liệt kê đường đi, vùng liên thông, quay lui |

Lỗi hay gặp: quên `visited` trên đồ thị có chu trình (lặp vô hạn); đệ quy quá sâu trên lưới lớn; ghi `out.append(duong)` thay vì bản sao `list(duong)` (về sau list bị sửa).

## 2.4 Sắp xếp tô-pô (topological sort) và phát hiện chu trình

### Bài toán

Cho đồ thị **có hướng**, cạnh `u → v` nghĩa là "`u` phải làm trước `v`". **Thứ tự tô-pô** là cách xếp mọi đỉnh thành dãy sao cho mọi cạnh đi từ **trước ra sau**. Tồn tại ⟺ đồ thị **không có chu trình có hướng** (là DAG).

```text
mặc áo lót → mặc áo sơ mi → thắt cà vạt → mặc vest
             mặc quần    → đi giày
thứ tự hợp lệ: áo lót, quần, sơ mi, giày, cà vạt, vest   (không duy nhất)
```

### Thuật toán Kahn (BFS theo bậc vào)

**Bậc vào** của `v` = số cạnh đi vào `v` = số việc phải làm trước `v`.

```text
q = các đỉnh bậc vào 0
while q: lấy u, ghi u; với mỗi u → v: indeg[v] -= 1, nếu về 0 thì thêm v vào q
nếu ghi được ít hơn n đỉnh → có chu trình
```

In [11]:
from collections import deque

def kahn_demo(n, edges):
    adj = [[] for _ in range(n)]
    indeg = [0] * n
    for u, v in edges:
        adj[u].append(v)
        indeg[v] += 1
    q = deque(i for i in range(n) if indeg[i] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order if len(order) == n else None

print("DAG:", kahn_demo(5, [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4)]))
print("có chu trình 1→2→3→1:", kahn_demo(4, [(0, 1), (1, 2), (2, 3), (3, 1)]))

DAG: [0, 1, 2, 3, 4]
có chu trình 1→2→3→1: None


### DFS ba màu

- **Trắng**: chưa thăm. **Xám**: đang trên đường đi đệ quy. **Đen**: đã xong.
- Gặp cạnh tới đỉnh **xám** → **chu trình**.
- Ghi đỉnh khi nó chuyển sang **đen** (thứ tự hậu tố), rồi **đảo ngược** → thứ tự tô-pô.

Vì sao đảo ngược? Với cạnh `u → v`: khi `u` xong thì `v` đã xong từ trước (hoặc đã đen sẵn) → `v` đứng trước `u` trong hậu tố → đảo lại thì `u` trước `v`.

In [12]:
def dfs_topo_demo(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
    color = [0] * n
    post = []
    def visit(u):
        color[u] = 1
        for v in adj[u]:
            if color[v] == 1:
                return False                   # chu trình
            if color[v] == 0 and not visit(v):
                return False
        color[u] = 2
        post.append(u)
        return True
    for u in range(n):
        if color[u] == 0 and not visit(u):
            return None
    return post[::-1]

print("DAG:", dfs_topo_demo(5, [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4)]))
print("có chu trình:", dfs_topo_demo(4, [(0, 1), (1, 2), (2, 3), (3, 1)]))

DAG: [0, 2, 1, 3, 4]
có chu trình: None


### Độ phức tạp

Cả hai cách: `O(V + E)` thời gian, `O(V + E)` bộ nhớ. Kahn không đệ quy nên an toàn với đồ thị dạng chuỗi dài.

### Lỗi hay gặp

- Chiều cạnh và bậc vào không nhất quán (`u → v` thì bậc vào tăng ở `v`).
- DFS chỉ dùng "đã thăm" (hai màu) → báo chu trình sai khi hai nhánh gặp lại cùng một đỉnh.
- Quên **đảo ngược** thứ tự hậu tố.
- Quên kiểm tra số đỉnh lấy ra ở Kahn → trả về thứ tự thiếu đỉnh khi có chu trình.

## 2.5 Áp dụng vào bài này

Môn học là **đỉnh**, `[a, b]` là **cạnh có hướng** `b → a` lưu bằng **danh sách kề**. Học hết được ⟺ đồ thị không có **chu trình có hướng** ⟺ tồn tại **thứ tự tô-pô**. Hai cách: **Kahn** (một dạng **BFS** theo bậc vào, đếm số đỉnh lấy ra) và **DFS ba màu** (gặp đỉnh xám là có chu trình).

# 3. Nền tảng

## Dựng đồ thị có hướng bằng danh sách kề

Cặp `[a, b]` ("học `b` trước `a`") ↦ cạnh `b → a`. Lưu `graph[b].append(a)` và đếm **bậc vào** `indeg[a] += 1` (số môn tiên quyết của `a`).

In [13]:
from typing import List
from collections import deque, defaultdict
import random, time

def build(num, prereq):
    graph = [[] for _ in range(num)]
    indeg = [0] * num
    for a, b in prereq:
        graph[b].append(a)
        indeg[a] += 1
    return graph, indeg

g, d = build(4, [[1, 0], [2, 0], [3, 1], [3, 2]])
print("danh sách kề (b → các môn mở khóa):", g)
print("bậc vào (số môn tiên quyết)       :", d)

danh sách kề (b → các môn mở khóa): [[1, 2], [3], [3], []]
bậc vào (số môn tiên quyết)       : [0, 1, 1, 2]


```text
      0
     / \          0 → 1 → 3
    1   2         0 → 2 → 3
     \ /
      3           thứ tự học hợp lệ: 0, 1, 2, 3   (hoặc 0, 2, 1, 3)
```

## Chu trình và DAG

**Chu trình** có hướng: đi theo các cạnh rồi quay lại đỉnh xuất phát (`0 → 1 → 0`). Trong chu trình, mỗi môn đều cần một môn khác trong chu trình học trước → không môn nào học được đầu tiên. Đồ thị có hướng **không** có chu trình gọi là **DAG**; mọi DAG đều có **thứ tự tô-pô** — sắp các đỉnh sao cho mọi cạnh đi từ trước ra sau.

# 4. Cách 1: thuật toán Kahn (BFS theo bậc vào)

Mô phỏng việc học thật:

```text
q = mọi môn có bậc vào 0 (không cần môn tiên quyết)
while q:
    lấy môn u ra, "học" nó (đếm taken += 1)
    với mỗi môn v mà u mở khóa: indeg[v] -= 1; nếu indeg[v] == 0 → đưa v vào q
return taken == numCourses
```

Nếu có chu trình, các môn trong chu trình **không bao giờ** về bậc vào 0 → `taken < numCourses`.

In [14]:
def can_finish_kahn(num: int, prereq: List[List[int]]) -> bool:
    graph, indeg = build(num, prereq)
    q = deque(i for i in range(num) if indeg[i] == 0)
    taken = 0
    while q:
        u = q.popleft()
        taken += 1
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return taken == num

print(can_finish_kahn(2, [[1, 0]]), can_finish_kahn(2, [[1, 0], [0, 1]]), can_finish_kahn(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))

True False True


# 5. Cách 2: DFS ba màu

Mỗi đỉnh có một trong ba trạng thái:

- **Trắng (0)**: chưa thăm.
- **Xám (1)**: **đang** thăm — nằm trên đường đi đệ quy hiện tại.
- **Đen (2)**: đã thăm xong (mọi đỉnh đi tới được từ nó đã xét, không có chu trình).

Đang DFS từ `u` mà gặp cạnh `u → v` với `v` **xám** ⟹ `v` là tổ tiên của `u` trên đường đi ⟹ **chu trình**. Gặp `v` đen thì bỏ qua (đã an toàn).

In [15]:
def can_finish_dfs(num: int, prereq: List[List[int]]) -> bool:
    graph, _ = build(num, prereq)
    color = [0] * num
    def has_cycle(u) -> bool:
        color[u] = 1
        for v in graph[u]:
            if color[v] == 1:
                return True                    # cạnh ngược → chu trình
            if color[v] == 0 and has_cycle(v):
                return True
        color[u] = 2
        return False
    return not any(color[u] == 0 and has_cycle(u) for u in range(num))

print(can_finish_dfs(2, [[1, 0]]), can_finish_dfs(2, [[1, 0], [0, 1]]), can_finish_dfs(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))

True False True


# 6. Mô phỏng Kahn

In [16]:
def trace_kahn(num, prereq):
    graph, indeg = build(num, prereq)
    q = deque(i for i in range(num) if indeg[i] == 0)
    order = []
    print("bậc vào ban đầu:", indeg, "| hàng đợi:", list(q))
    while q:
        u = q.popleft()
        order.append(u)
        freed = []
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v); freed.append(v)
        print(f"   học {u}: bậc vào = {indeg}, mở khóa {freed}, hàng đợi {list(q)}")
    print("thứ tự học:", order, "| học hết?", len(order) == num)

trace_kahn(4, [[1, 0], [2, 0], [3, 1], [3, 2]])
trace_kahn(4, [[1, 0], [2, 1], [1, 2], [3, 0]])

bậc vào ban đầu: [0, 1, 1, 2] | hàng đợi: [0]
   học 0: bậc vào = [0, 0, 0, 2], mở khóa [1, 2], hàng đợi [1, 2]
   học 1: bậc vào = [0, 0, 0, 1], mở khóa [], hàng đợi [2]
   học 2: bậc vào = [0, 0, 0, 0], mở khóa [3], hàng đợi [3]
   học 3: bậc vào = [0, 0, 0, 0], mở khóa [], hàng đợi []
thứ tự học: [0, 1, 2, 3] | học hết? True
bậc vào ban đầu: [0, 2, 1, 1] | hàng đợi: [0]
   học 0: bậc vào = [0, 1, 1, 0], mở khóa [3], hàng đợi [3]
   học 3: bậc vào = [0, 1, 1, 0], mở khóa [], hàng đợi []
thứ tự học: [0, 3] | học hết? False


Ví dụ thứ hai: `1` và `2` cần nhau (chu trình `1 → 2 → 1`), bậc vào của chúng không bao giờ về 0 → chỉ học được `0, 3`.

# 7. Phiên bản đúng format LeetCode

Dùng Kahn (không đệ quy → không lo độ sâu với 2000 đỉnh xếp thành chuỗi).

In [17]:
class Solution:
    def canFinish(self, numCourses: int, prerequisites: List[List[int]]) -> bool:
        graph = [[] for _ in range(numCourses)]
        indeg = [0] * numCourses
        for a, b in prerequisites:
            graph[b].append(a)
            indeg[a] += 1
        q = deque(i for i in range(numCourses) if indeg[i] == 0)
        taken = 0
        while q:
            u = q.popleft()
            taken += 1
            for v in graph[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        return taken == numCourses

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


def has_cycle_brute(num, prereq):
    """Có chu trình ⟺ tồn tại cạnh b → a mà a đi tới được b (kiểm tra bằng bao đóng bắc cầu)."""
    reach = [[False] * num for _ in range(num)]
    for a, b in prereq:
        reach[b][a] = True
    for k in range(num):
        for i in range(num):
            if reach[i][k]:
                for j in range(num):
                    if reach[k][j]:
                        reach[i][j] = True
    return any(reach[i][i] for i in range(num))


sol = Solution()
check("Ví dụ 1", sol.canFinish(2, [[1, 0]]), True)
check("Ví dụ 2", sol.canFinish(2, [[1, 0], [0, 1]]), False)
check("Không cạnh", sol.canFinish(3, []), True)
check("Kim cương", sol.canFinish(4, [[1, 0], [2, 0], [3, 1], [3, 2]]), True)
check("Vòng 3 đỉnh", sol.canFinish(3, [[1, 0], [2, 1], [0, 2]]), False)

random.seed(207)
for _ in range(2000):
    n = random.randint(1, 7)
    pairs = {(random.randrange(n), random.randrange(n)) for _ in range(random.randint(0, 10))}
    prereq = [[a, b] for a, b in pairs if a != b]
    exp = not has_cycle_brute(n, prereq)
    assert sol.canFinish(n, prereq) == exp == can_finish_dfs(n, prereq), (n, prereq)
print("2000 đồ thị ngẫu nhiên: Kahn và DFS khớp kiểm tra bằng bao đóng bắc cầu")

chain = [[i + 1, i] for i in range(1999)]
t0 = time.time(); r = sol.canFinish(2000, chain); t1 = time.time()
print(f"chuỗi 2000 môn: {r}, {t1 - t0:.4f}s")

Ví dụ 1          PASS  True
Ví dụ 2          PASS  False
Không cạnh       PASS  True
Kim cương        PASS  True
Vòng 3 đỉnh      PASS  False


2000 đồ thị ngẫu nhiên: Kahn và DFS khớp kiểm tra bằng bao đóng bắc cầu


chuỗi 2000 môn: True, 0.0007s


# 8. Vì sao đúng? Độ phức tạp?

**Kahn:** một đỉnh vào hàng đợi khi mọi môn tiên quyết của nó đã được "học". Nếu đồ thị là DAG thì luôn còn ít nhất một đỉnh bậc vào 0 trong phần chưa học (một DAG khác rỗng luôn có đỉnh không có cạnh đi vào) → học được hết. Nếu có chu trình, không đỉnh nào của chu trình về bậc vào 0 trước các đỉnh khác của chu trình → không bao giờ vào hàng đợi.

**DFS ba màu:** có chu trình ⟺ DFS gặp **cạnh ngược** (tới đỉnh xám). Đỉnh đen không thể nằm trên chu trình đi qua đỉnh đang xét (mọi đỉnh tới được từ nó đã xét xong mà không gặp chu trình).

```text
Time:  O(V + E)     (V = numCourses, E = len(prerequisites))
Space: O(V + E)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: DFS hai màu (chỉ "đã thăm") — báo chu trình sai

In [19]:
def wrong_two_colors(num, prereq):
    graph, _ = build(num, prereq)
    visited = [False] * num
    def dfs(u):
        visited[u] = True
        for v in graph[u]:
            if visited[v]:
                return True                       # SAI: v có thể đã thăm xong ở nhánh khác
            if dfs(v):
                return True
        return False
    return not any(not visited[u] and dfs(u) for u in range(num))

diamond = [[1, 0], [2, 0], [3, 1], [3, 2]]
print("Kim cương — sai:", wrong_two_colors(4, diamond), "| đúng:", sol.canFinish(4, diamond))

Kim cương — sai: False | đúng: True


Đỉnh `3` được tới qua `0 → 1 → 3` rồi lại qua `0 → 2 → 3`: lần thứ hai nó đã "thăm" nhưng **không** nằm trên đường đi hiện tại — không phải chu trình. Cần phân biệt **đang thăm** (xám) và **đã xong** (đen).

## Lỗi 2: Quên đặt màu đen khi DFS xong

In [20]:
def wrong_no_black(num, prereq):
    graph, _ = build(num, prereq)
    color = [0] * num
    def has_cycle(u):
        color[u] = 1
        for v in graph[u]:
            if color[v] == 1:
                return True
            if color[v] == 0 and has_cycle(v):
                return True
        return False                              # SAI: u vẫn xám mãi
    return not any(color[u] == 0 and has_cycle(u) for u in range(num))

print("Kim cương — sai:", wrong_no_black(4, diamond), "| đúng:", sol.canFinish(4, diamond))

Kim cương — sai: False | đúng: True


## Lỗi 3: Dựng cạnh ngược chiều và đếm bậc vào sai

In [21]:
def wrong_direction(num, prereq):
    graph = [[] for _ in range(num)]
    indeg = [0] * num
    for a, b in prereq:
        graph[a].append(b)                        # cạnh a → b
        indeg[a] += 1                             # SAI: bậc vào phải tăng ở đỉnh cuối cạnh (b)
    q = deque(i for i in range(num) if indeg[i] == 0)
    taken = 0
    while q:
        u = q.popleft(); taken += 1
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return taken == num

print("Ví dụ 1 — sai:", wrong_direction(2, [[1, 0]]), "| đúng:", sol.canFinish(2, [[1, 0]]))

Ví dụ 1 — sai: False | đúng: True


Chiều cạnh nào cũng được (đồ thị đảo chiều có chu trình ⟺ đồ thị gốc có), nhưng **bậc vào** phải tính ở **đích** của cạnh đã chọn. Trộn hai quy ước là lỗi phổ biến.

## Lỗi 4: DFS đệ quy với chuỗi dài

Chuỗi 2000 môn → đệ quy sâu 2000 tầng, vượt giới hạn mặc định (~1000). Kahn hoặc DFS bằng ngăn xếp tránh được.

In [22]:
import sys
old = sys.getrecursionlimit(); sys.setrecursionlimit(1000)
try:
    can_finish_dfs(2000, chain)
    print("DFS đệ quy chạy được")
except RecursionError:
    print("RecursionError với chuỗi 2000 môn (DFS đệ quy)")
finally:
    sys.setrecursionlimit(old)

RecursionError với chuỗi 2000 môn (DFS đệ quy)


# 10. Mẹo nhận diện pattern

- **"A phải trước B", phụ thuộc, thứ tự thực hiện** → đồ thị có hướng + **sắp xếp tô-pô**.
- **"Có làm được hết không?"** = "có chu trình không?" → Kahn (đếm số đỉnh lấy ra) hoặc DFS ba màu.
- **Cần một thứ tự cụ thể** → thứ tự lấy ra ở Kahn, hoặc **đảo ngược** thứ tự hậu tố ở DFS (#210).

Bài liên quan: **#210 Course Schedule II**, **#269 Alien Dictionary** (Hard), **#802 Find Eventual Safe States**, **#1462 Course Schedule IV**, **#2050 Parallel Courses III** (Hard), **#329 Longest Increasing Path in a Matrix** (Hard).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `numCourses = 5`, `prerequisites = [[1, 0], [2, 1], [3, 2], [1, 3], [4, 0]]`, chạy Kahn và cho biết môn nào không học được.

## Bài 2 — Điền chỗ trống (Kahn)

In [23]:
def exercise_2_can_finish(numCourses: int, prerequisites: List[List[int]]) -> bool:
    graph = [[] for _ in range(numCourses)]
    indeg = [0] * numCourses
    for a, b in prerequisites:
        graph[b].append(a)
        indeg[a] += 1
    # TODO: BFS các đỉnh bậc vào 0, đếm số môn học được
    pass


# assert exercise_2_can_finish(2, [[1, 0]]) is True
# assert exercise_2_can_finish(2, [[1, 0], [0, 1]]) is False
# assert exercise_2_can_finish(4, [[1, 0], [2, 0], [3, 1], [3, 2]]) is True

## Bài 3 — Viết bản DFS ba màu **không đệ quy** (ngăn xếp)

Gợi ý: ngăn xếp chứa cặp `(đỉnh, chỉ số cạnh kế tiếp cần xét)`.

In [24]:
def exercise_3_can_finish_iter(numCourses: int, prerequisites: List[List[int]]) -> bool:
    # TODO
    pass


# assert exercise_3_can_finish_iter(2, [[1, 0], [0, 1]]) is False
# assert exercise_3_can_finish_iter(4, [[1, 0], [2, 0], [3, 1], [3, 2]]) is True
# assert exercise_3_can_finish_iter(2000, [[i + 1, i] for i in range(1999)]) is True

## Bài 4 — Biến thể: trả về **một chu trình** (danh sách đỉnh theo chiều cạnh, đỉnh đầu không lặp lại), hoặc `[]` nếu không có

In [25]:
def exercise_4_find_cycle(num: int, prereq: List[List[int]]) -> List[int]:
    # TODO
    pass


def is_cycle(cyc, prereq):
    edges = {(b, a) for a, b in prereq}               # cạnh b → a
    return len(cyc) > 0 and all((cyc[i], cyc[(i + 1) % len(cyc)]) in edges for i in range(len(cyc)))

# assert is_cycle(exercise_4_find_cycle(4, [[1, 0], [2, 1], [1, 2], [3, 0]]), [[1, 0], [2, 1], [1, 2], [3, 0]])
# assert exercise_4_find_cycle(3, [[1, 0], [2, 1]]) == []

## Bài 5 — Mini challenge: Course Schedule IV (LeetCode #1462)

Đồ thị là DAG. Với mỗi truy vấn `[u, v]`, trả lời "`u` có phải môn tiên quyết (trực tiếp hoặc gián tiếp) của `v` không?". Gợi ý: duyệt theo thứ tự tô-pô, mỗi đỉnh giữ **tập** (hoặc bitmask) các tổ tiên của nó.

In [26]:
def exercise_5_check_if_prerequisite(numCourses: int, prerequisites: List[List[int]], queries: List[List[int]]) -> List[bool]:
    # TODO: lưu ý ở bài này cặp [u, v] nghĩa là "u là tiên quyết của v"
    pass


# assert exercise_5_check_if_prerequisite(2, [[1, 0]], [[0, 1], [1, 0]]) == [False, True]
# assert exercise_5_check_if_prerequisite(3, [[1, 2], [1, 0], [2, 0]], [[1, 0], [1, 2]]) == [True, True]
# assert exercise_5_check_if_prerequisite(2, [], [[1, 0], [0, 1]]) == [False, False]

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_kahn(5, ...)` sau khi tự làm.

**Bài 3**
- Mức 1: với mỗi đỉnh trắng `s`: tô xám, đẩy `(s, 0)`.
- Mức 2: nhìn đỉnh `(u, i)` trên đỉnh ngăn xếp: nếu `i < len(graph[u])` thì tăng `i`, xét `v = graph[u][i]` (xám → chu trình; trắng → tô xám, đẩy `(v, 0)`); nếu hết cạnh thì tô đen và pop.

**Bài 4**
- Mức 1: DFS ba màu, giữ `parent` hoặc ngăn xếp đường đi; khi gặp cạnh `u → v` với `v` xám, chu trình là đoạn đường đi từ `v` tới `u`.

**Bài 5**
- Mức 1: cạnh `u → v` (u tiên quyết của v). Kahn; khi lấy `u` ra, với mỗi `v` kề: `anc[v] |= anc[u] | (1 << u)`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [27]:
trace_kahn(5, [[1, 0], [2, 1], [3, 2], [1, 3], [4, 0]])

bậc vào ban đầu: [0, 2, 1, 1, 1] | hàng đợi: [0]
   học 0: bậc vào = [0, 1, 1, 1, 0], mở khóa [4], hàng đợi [4]
   học 4: bậc vào = [0, 1, 1, 1, 0], mở khóa [], hàng đợi []
thứ tự học: [0, 4] | học hết? False


## Đáp án Bài 2

In [28]:
def answer_2_can_finish(numCourses: int, prerequisites: List[List[int]]) -> bool:
    return Solution().canFinish(numCourses, prerequisites)

assert answer_2_can_finish(2, [[1, 0]]) is True
assert answer_2_can_finish(2, [[1, 0], [0, 1]]) is False
assert answer_2_can_finish(4, [[1, 0], [2, 0], [3, 1], [3, 2]]) is True
print("Bài 2 PASS (giống Solution ở mục 7)")

Bài 2 PASS (giống Solution ở mục 7)

## Đáp án Bài 3

In [29]:
def answer_3_can_finish_iter(numCourses: int, prerequisites: List[List[int]]) -> bool:
    graph, _ = build(numCourses, prerequisites)
    color = [0] * numCourses
    for s in range(numCourses):
        if color[s]:
            continue
        color[s] = 1
        st = [[s, 0]]
        while st:
            u, i = st[-1]
            if i < len(graph[u]):
                st[-1][1] += 1
                v = graph[u][i]
                if color[v] == 1:
                    return False
                if color[v] == 0:
                    color[v] = 1
                    st.append([v, 0])
            else:
                color[u] = 2
                st.pop()
    return True

assert answer_3_can_finish_iter(2, [[1, 0], [0, 1]]) is False
assert answer_3_can_finish_iter(4, [[1, 0], [2, 0], [3, 1], [3, 2]]) is True
assert answer_3_can_finish_iter(2000, [[i + 1, i] for i in range(1999)]) is True
for _ in range(2000):
    n = random.randint(1, 7)
    pairs = {(random.randrange(n), random.randrange(n)) for _ in range(random.randint(0, 10))}
    prereq = [[a, b] for a, b in pairs if a != b]
    assert answer_3_can_finish_iter(n, prereq) == sol.canFinish(n, prereq)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [30]:
def answer_4_find_cycle(num: int, prereq: List[List[int]]) -> List[int]:
    graph, _ = build(num, prereq)
    color = [0] * num
    path = []
    def dfs(u):
        color[u] = 1
        path.append(u)
        for v in graph[u]:
            if color[v] == 1:
                return path[path.index(v):]          # đoạn từ v tới u
            if color[v] == 0:
                found = dfs(v)
                if found:
                    return found
        color[u] = 2
        path.pop()
        return None
    for s in range(num):
        if color[s] == 0:
            found = dfs(s)
            if found:
                return list(found)
    return []

assert is_cycle(answer_4_find_cycle(4, [[1, 0], [2, 1], [1, 2], [3, 0]]), [[1, 0], [2, 1], [1, 2], [3, 0]])
assert answer_4_find_cycle(3, [[1, 0], [2, 1]]) == []
for _ in range(2000):
    n = random.randint(1, 7)
    pairs = {(random.randrange(n), random.randrange(n)) for _ in range(random.randint(0, 10))}
    prereq = [[a, b] for a, b in pairs if a != b]
    cyc = answer_4_find_cycle(n, prereq)
    assert (cyc == []) == sol.canFinish(n, prereq) and (cyc == [] or is_cycle(cyc, prereq))
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [31]:
def answer_5_check_if_prerequisite(numCourses: int, prerequisites: List[List[int]], queries: List[List[int]]) -> List[bool]:
    graph = [[] for _ in range(numCourses)]
    indeg = [0] * numCourses
    for u, v in prerequisites:                     # u là tiên quyết của v: cạnh u → v
        graph[u].append(v)
        indeg[v] += 1
    anc = [0] * numCourses                         # bitmask các tổ tiên
    q = deque(i for i in range(numCourses) if indeg[i] == 0)
    while q:
        u = q.popleft()
        for v in graph[u]:
            anc[v] |= anc[u] | (1 << u)
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return [bool(anc[v] >> u & 1) for u, v in queries]

assert answer_5_check_if_prerequisite(2, [[1, 0]], [[0, 1], [1, 0]]) == [False, True]
assert answer_5_check_if_prerequisite(3, [[1, 2], [1, 0], [2, 0]], [[1, 0], [1, 2]]) == [True, True]
assert answer_5_check_if_prerequisite(2, [], [[1, 0], [0, 1]]) == [False, False]
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Môn học = đỉnh, `[a, b]` = cạnh `b → a`. Học hết ⟺ không có chu trình. **Kahn**: đưa các đỉnh bậc vào 0 vào hàng đợi, lấy ra thì giảm bậc vào của hàng xóm; đếm số đỉnh lấy ra có bằng `numCourses` không. **DFS ba màu**: gặp đỉnh **xám** (đang trên đường đi) là có chu trình. `O(V + E)`.

---

## Checklist trước khi nộp

- [ ] Chiều cạnh và bậc vào nhất quán?
- [ ] Kahn: so số môn lấy ra với `numCourses`?
- [ ] DFS: ba trạng thái, tô đen khi xong?
- [ ] Không đệ quy quá sâu (2000 đỉnh)?